# W03 — Data Contract

**Lane:** `writing-data-contracts`  
**Dataset:** `flyrank/flyrank-data`  
**Main Table:** `fact_content_daily_performance`

## Goal
- Define a data contract.
- Verify the contract with SQL.
- Build an honest feature frame.
- Demonstrate data leakage.
- State limitations.


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

month = "2026-03"
display(Markdown(f"**Working Month:** {month}"))


# 1. Data Contract

## Unit of Analysis
One row represents **one content item for one client on one report date**.

**Primary Grain**

`(report_date, client_id, content_id)`

## Time Window

This notebook analyzes **March 2026**, a mid-panel month.

## Field Classification

### Features
- content_age_days
- prev_7d_impressions
- prev_7d_clicks
- click_through_rate_prev7d
- client_content_share

### Label
- target

### Context
- report_date
- client_id
- content_id

### Excluded
- trend_pct
- trend_direction
- is_declining_label
- Future-looking columns

## Missing Values

Missing values should be checked overall and by category because missingness may follow content type rather than being random.

## Output

A verified feature dataset suitable for downstream ranking or prediction without leakage.


## 2. Verification Query 1 — Grain

In [ ]:
query_grain = f'''
SELECT
  report_date,
  client_id,
  content_id,
  COUNT(*) AS rows_per_key
FROM fact_content_daily_performance
WHERE month = "{month}"
GROUP BY report_date, client_id, content_id
HAVING COUNT(*) > 1
LIMIT 20;
'''
print(query_grain)


## 3. Verification Query 2 — Row Count & Date Window

In [ ]:
query_window = f'''
SELECT
  COUNT(*) AS row_count,
  MIN(report_date) AS min_date,
  MAX(report_date) AS max_date
FROM fact_content_daily_performance
WHERE month="{month}";
'''
print(query_window)


## 4. Verification Query 3 — Missingness

In [ ]:
query_missing = f'''
SELECT
AVG(CASE WHEN publish_date IS NULL THEN 1.0 ELSE 0 END) AS publish_missing,
AVG(CASE WHEN prev_7d_impressions IS NULL THEN 1.0 ELSE 0 END) AS impressions_missing,
AVG(CASE WHEN prev_7d_clicks IS NULL THEN 1.0 ELSE 0 END) AS clicks_missing
FROM fact_content_daily_performance
WHERE month="{month}";
'''
print(query_missing)


# 5. Honest Features

| Feature | Why Safe |
|---------|----------|
| content_age_days | Known at prediction time |
| prev_7d_impressions | Uses only historical data |
| prev_7d_clicks | Uses only historical data |
| click_through_rate_prev7d | Computed from past observations |
| client_content_share | Historical client statistics |


In [ ]:
query_features = f'''
WITH base AS (
SELECT *
FROM fact_content_daily_performance
WHERE month="{month}"
AND is_available IS TRUE
)
SELECT
report_date,
client_id,
content_id,
DATE_DIFF(report_date,publish_date,DAY) AS content_age_days,
prev_7d_impressions,
prev_7d_clicks,
SAFE_DIVIDE(prev_7d_clicks,prev_7d_impressions) AS click_through_rate_prev7d,
client_content_share,
target
FROM base
LIMIT 10;
'''
print(query_features)


# 6. Leakage Demonstration

The following example intentionally adds the label as a feature. A model trained this way would achieve unrealistically high performance because it can directly "see" the answer.


In [ ]:
# Example only (requires a loaded dataframe named df)

# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import r2_score
#
# X = df[['content_age_days','prev_7d_impressions',
#         'prev_7d_clicks','click_through_rate_prev7d',
#         'client_content_share']]
# y = df['target']
#
# X['leak'] = y   # BAD PRACTICE
#
# Train once with leak, then remove:
# X = X.drop(columns=['leak'])


# 7. Limitation

- Only one month (March 2026) is analyzed.
- Client history depth differs across the panel.
- Seasonal effects are not captured.
- Results should not be generalized without validating on additional months.


# 8. Self Check

- ✅ Unit of analysis defined
- ✅ Time window defined
- ✅ Field classification completed
- ✅ Grain verified
- ✅ Row count verified
- ✅ Date window verified
- ✅ Missingness query included
- ✅ Honest features listed
- ✅ Leakage demonstrated
- ✅ Limitation stated
